In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [9]:
# Load dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [10]:
# Constants
TRAIN_SIZE = 0.2
RANDOMSTATE = 1082

In [11]:
# Preprocess data
# Clean column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

X = df.drop(columns=['timestamp', 'label'])
y = df['label']

# Encode labels
if y.dtype == object or y.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(y)
    
# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split benign and malicious
benign_mask = y == 0
X_benign = X_scaled[benign_mask]
y_benign = y[benign_mask]

# Training set: benign only
X_train, X_benign_val, y_train, y_benign_val = train_test_split(
    X_benign, y_benign, train_size=TRAIN_SIZE, random_state=RANDOMSTATE
)

# Test set: all samples (benign and malicious)
X_test = X_scaled
y_test = y

In [12]:
# Train One-Class SVM on benign only
ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.04)  # nu ≈ expected outlier ratio
ocsvm.fit(X_train)

OneClassSVM(gamma='auto', nu=0.04)

In [13]:
# Predict: +1 for inliers (benign), -1 for outliers (anomaly)
y_pred_ocsvm = ocsvm.predict(X_test)
# Convert to 0=benign, 1=anomaly
y_pred = (y_pred_ocsvm == -1).astype(int)

In [14]:
# Results
print(f"Model Accuracy on Test Set: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Model Accuracy on Test Set: 95.42%
Confusion Matrix:
[[2966  158]
 [   8  490]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      3124
           1       0.76      0.98      0.86       498

    accuracy                           0.95      3622
   macro avg       0.88      0.97      0.91      3622
weighted avg       0.96      0.95      0.96      3622

